# Parte 2: Clasificación de Texto con RNN y LSTM
## Dataset: IMDB — Análisis de Sentimientos en Reseñas de Películas

**Dataset:** 50,000 reseñas de películas en inglés (25,000 train / 25,000 test), 2 clases: positivo y negativo.  
**Objetivo:** Clasificar el sentimiento de reseñas usando primero una **RNN simple** y luego una **LSTM**, comparando ambas arquitecturas.

El texto se procesa como **secuencia de palabras**: cada reseña es una serie temporal donde el orden importa. Esto motiva el uso de redes recurrentes (RNN/LSTM) en lugar de modelos clásicos de ML.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

tf.random.set_seed(42)
np.random.seed(42)

print('TensorFlow:', tf.__version__)

---
## i. Conjunto de Datos: IMDB

El dataset **IMDB** contiene reseñas de películas etiquetadas como:
- **0 → Negativa:** el crítico expresa una opinión desfavorable sobre la película.
- **1 → Positiva:** el crítico expresa una opinión favorable sobre la película.

Viene **pre-tokenizado** en Keras: cada palabra está reemplazada por un índice entero según su frecuencia en el corpus. Usaremos las **10,000 palabras más frecuentes** (`num_words=10000`) para construir el vocabulario, ignorando palabras raras que no aportan información suficiente.

In [ ]:
NUM_WORDS = 10_000   # tamaño del vocabulario

print("Cargando dataset IMDB...")
(X_train_raw, y_train), (X_test_raw, y_test) = imdb.load_data(num_words=NUM_WORDS)

print(f"Train : {len(X_train_raw):,} reseñas")
print(f"Test  : {len(X_test_raw):,}  reseñas")
print(f"Clases: 0=Negativa | 1=Positiva")
print(f"Balance train: {np.mean(y_train)*100:.1f}% positivas")
print(f"Vocabulario   : {NUM_WORDS:,} palabras")

In [ ]:
# Decodificar algunas reseñas para mostrar ejemplos legibles
word_index   = imdb.get_word_index()
reverse_index = {v+3: k for k, v in word_index.items()}
reverse_index.update({0: '<PAD>', 1: '<START>', 2: '<UNK>', 3: '<UNUSED>'})

def decode_review(encoded):
    return ' '.join(reverse_index.get(i, '?') for i in encoded)

print("=" * 70)
print("EJEMPLOS DEL DATASET")
print("=" * 70)
for i in range(4):
    texto   = decode_review(X_train_raw[i])
    clase   = 'POSITIVA' if y_train[i] == 1 else 'NEGATIVA'
    palabras = len(X_train_raw[i])
    print(f"\n[Reseña {i+1}] Clase: {clase} | Palabras: {palabras}")
    print(f"  {texto[:300]}...")
    print("-" * 70)

In [ ]:
# Distribución de longitudes de reseñas
longitudes = [len(x) for x in X_train_raw]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(longitudes, bins=50, color='#3498db', edgecolor='white')
axes[0].axvline(np.median(longitudes), color='#e74c3c', linestyle='--',
                label=f'Mediana: {np.median(longitudes):.0f}')
axes[0].axvline(np.percentile(longitudes, 90), color='#e67e22', linestyle=':',
                label=f'P90: {np.percentile(longitudes, 90):.0f}')
axes[0].set_xlabel('Longitud (palabras)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de longitudes de reseñas')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Balance de clases
clases, conteos = np.unique(y_train, return_counts=True)
nombres_clase   = ['Negativa', 'Positiva']
axes[1].bar(nombres_clase, conteos, color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1.5)
for i, c in enumerate(conteos):
    axes[1].text(i, c + 100, f'{c:,}', ha='center', fontweight='bold')
axes[1].set_ylabel('Número de reseñas')
axes[1].set_title('Balance de clases (train)')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('distribucion_dataset_texto.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Longitud promedio : {np.mean(longitudes):.0f} palabras")
print(f"Longitud mediana  : {np.median(longitudes):.0f} palabras")
print(f"Percentil 90      : {np.percentile(longitudes, 90):.0f} palabras")
print(f"Máximo            : {max(longitudes)} palabras")

---
## ii. Preprocesamiento

**Pasos:**

1. **Vocabulario:** ya construido por Keras con las 10,000 palabras más frecuentes. Palabras fuera del vocabulario se convierten en `<UNK>`. El tamaño del vocabulario controla el balance entre cobertura del lenguaje y la dimensión del embedding.

2. **Longitud máxima (`MAX_LEN`):** el 90% de las reseñas tiene menos de ~600 palabras. Usaremos `MAX_LEN=300` para cubrir la mediana (~240 palabras) sin que el padding infle excesivamente las secuencias cortas.

3. **Padding:** las secuencias más cortas que MAX_LEN se rellenan con ceros al final (`post`), y las más largas se truncan al inicio (`pre`) para conservar el final de la reseña donde suelen estar las conclusiones de la crítica.

In [ ]:
MAX_LEN = 300   # longitud máxima de secuencia

X_train = pad_sequences(X_train_raw, maxlen=MAX_LEN, padding='post', truncating='pre')
X_test  = pad_sequences(X_test_raw,  maxlen=MAX_LEN, padding='post', truncating='pre')

print("Preprocesamiento completado:")
print(f"  Vocabulario (NUM_WORDS) : {NUM_WORDS:,}")
print(f"  Longitud máxima         : {MAX_LEN} tokens")
print(f"  X_train shape           : {X_train.shape}")
print(f"  X_test  shape           : {X_test.shape}")
print(f"  Tipo de datos           : {X_train.dtype}")
print()
print("Cada fila es una reseña representada como secuencia de enteros")
print("Ejemplo (primeros 20 tokens):", X_train[0, :20])

---
## iii. Modelo 1 — RNN Simple

**Arquitectura:** `Embedding → SimpleRNN → Dense(softmax)`

- **Embedding (10000 → 64 dims):** convierte cada índice entero en un vector denso de 64 dimensiones. La capa aprende representaciones semánticas durante el entrenamiento (palabras similares → vectores cercanos). Elegimos 64 dims como balance entre capacidad expresiva y velocidad de entrenamiento.

- **SimpleRNN (64 unidades):** procesa la secuencia token a token, manteniendo un estado oculto que "recuerda" el contexto anterior. Limitación: el gradiente puede desvanecerse en secuencias largas (problema de larga dependencia).

- **Dense(1, sigmoid):** capa de salida binaria (positivo/negativo).

Se prueban dos configuraciones de hiperparámetros: número de unidades en la RNN (32 vs 64).

In [ ]:
EMBED_DIM   = 64
BATCH_SIZE  = 64
EPOCHS_RNN  = 10

def build_rnn(units=64, embed_dim=EMBED_DIM):
    model = keras.Sequential([
        layers.Embedding(NUM_WORDS, embed_dim, input_length=MAX_LEN),
        layers.SimpleRNN(units),
        layers.Dense(1, activation='sigmoid')
    ], name=f'RNN_{units}units')
    return model

# Mostrar arquitectura del modelo con 64 unidades
rnn_demo = build_rnn(64)
rnn_demo.summary()

### Hiperparámetros probados en RNN

| Config | Unidades SimpleRNN | Learning Rate | Justificación |
|--------|--------------------|---------------|---------------|
| RNN-A  | 32                 | 0.001         | Modelo más ligero, menos parámetros, más rápido |
| RNN-B  | 64                 | 0.001         | Mayor capacidad de memoria recurrente |

In [ ]:
def entrenar(model, nombre, epochs=EPOCHS_RNN):
    model.compile(
        optimizer=keras.optimizers.Adam(0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    t0 = time.time()
    hist = model.fit(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=epochs,
        validation_split=0.20,
        verbose=1
    )
    elapsed = time.time() - t0
    print(f"\n{nombre} entrenado en {elapsed:.0f}s")
    return model, hist, elapsed

print("=== RNN Config-A: 32 unidades ===")
rnn_A, hist_rnnA, t_rnnA = entrenar(build_rnn(32), 'RNN-A (32 units)')

In [ ]:
print("=== RNN Config-B: 64 unidades ===")
rnn_B, hist_rnnB, t_rnnB = entrenar(build_rnn(64), 'RNN-B (64 units)')

In [ ]:
def plot_historia(hist, nombre, color='#3498db'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'Curvas de entrenamiento — {nombre}', fontsize=13, fontweight='bold')
    for ax, met, lbl in zip(axes, ['accuracy', 'loss'], ['Exactitud', 'Pérdida']):
        ax.plot(hist.history[met],         label='Train', color=color, linewidth=2)
        ax.plot(hist.history[f'val_{met}'],label='Val',   color=color, linestyle='--', linewidth=2)
        ax.set_xlabel('Época'); ax.set_ylabel(lbl)
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'curvas_{nombre.replace(" ","_")}.png', dpi=100, bbox_inches='tight')
    plt.show()

plot_historia(hist_rnnA, 'RNN-A 32 units', '#3498db')
plot_historia(hist_rnnB, 'RNN-B 64 units', '#9b59b6')

In [ ]:
# Seleccionar la mejor RNN
va_rnnA = max(hist_rnnA.history['val_accuracy'])
va_rnnB = max(hist_rnnB.history['val_accuracy'])

print(f"RNN-A (32 units): mejor val_acc = {va_rnnA:.4f}")
print(f"RNN-B (64 units): mejor val_acc = {va_rnnB:.4f}")

best_rnn  = rnn_A if va_rnnA >= va_rnnB else rnn_B
best_rnn_name = 'RNN-A (32 units)' if va_rnnA >= va_rnnB else 'RNN-B (64 units)'
print(f"\nMejor RNN: {best_rnn_name}")

---
## iv. Entrenamiento RNN — Selección del número de épocas

Se reentrena la mejor configuración RNN con distintos números de épocas para identificar el punto de ajuste óptimo y observar si aparece sobreajuste.

In [ ]:
EPOCH_LIST_RNN = [3, 5, 10, 15, 20]
best_units_rnn = 32 if va_rnnA >= va_rnnB else 64

resultados_rnn = []
print(f"{'Épocas':>7} | train_acc | val_acc | val_loss")
print("-" * 45)

historiales_rnn = []
for ep in EPOCH_LIST_RNN:
    m = build_rnn(best_units_rnn)
    m.compile(optimizer=keras.optimizers.Adam(0.001),
              loss='binary_crossentropy', metrics=['accuracy'])
    hist = m.fit(X_train, y_train, batch_size=BATCH_SIZE,
                 epochs=ep, validation_split=0.20, verbose=0)
    tr  = hist.history['accuracy'][-1]
    va  = hist.history['val_accuracy'][-1]
    vl  = hist.history['val_loss'][-1]
    print(f"{ep:>7} | {tr:.4f}    | {va:.4f}  | {vl:.4f}")
    historiales_rnn.append((ep, hist))
    resultados_rnn.append({'epochs': ep, 'train_acc': tr, 'val_acc': va})

In [ ]:
# Curvas completas de todos los conteos de épocas RNN
fig, axes = plt.subplots(len(EPOCH_LIST_RNN), 2, figsize=(13, 4 * len(EPOCH_LIST_RNN)))
fig.suptitle('RNN — Curvas de aprendizaje por número de épocas', fontsize=14, fontweight='bold')

for idx, (ep, hist) in enumerate(historiales_rnn):
    for ax, met, lbl in zip(axes[idx], ['accuracy', 'loss'], ['Exactitud', 'Pérdida']):
        ax.plot(hist.history[met],          label='Train', color='#3498db', linewidth=2)
        ax.plot(hist.history[f'val_{met}'], label='Val',   color='#e74c3c', linewidth=2, linestyle='--')
        ax.set_title(f'RNN {ep} épocas — {lbl}', fontsize=10)
        ax.set_xlabel('Época'); ax.set_ylabel(lbl)
        ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_rnn_epocas.png', dpi=100, bbox_inches='tight')
plt.show()

idx_best_rnn   = int(np.argmax([r['val_acc'] for r in resultados_rnn]))
BEST_EP_RNN    = resultados_rnn[idx_best_rnn]['epochs']
print(f"\nMejor número de épocas RNN: {BEST_EP_RNN}")

---
## v. Modelo 2 — LSTM

**Arquitectura:** `Embedding → LSTM → Dense(sigmoid)`

Se usa **el mismo embedding (10000 → 64 dims) y la misma longitud MAX_LEN=300** que en la RNN para que la comparación sea justa: la única diferencia es la capa recurrente.

**¿Por qué LSTM sobre SimpleRNN?**  
La LSTM tiene compuertas (forget, input, output) que controlan qué información se retiene o descarta del estado oculto. Esto resuelve el **problema de desvanecimiento del gradiente** de la RNN simple, permitiendo aprender dependencias de largo alcance (ej: "aunque al principio parecía buena... al final fue terrible").

Se prueban dos configuraciones: 32 vs 64 unidades LSTM para comparación justa con la RNN.

In [ ]:
def build_lstm(units=64, embed_dim=EMBED_DIM):
    model = keras.Sequential([
        layers.Embedding(NUM_WORDS, embed_dim, input_length=MAX_LEN),
        layers.LSTM(units),
        layers.Dense(1, activation='sigmoid')
    ], name=f'LSTM_{units}units')
    return model

lstm_demo = build_lstm(64)
lstm_demo.summary()
print("\nNota: la LSTM tiene ~4x más parámetros que la SimpleRNN equivalente")
print("por sus 4 compuertas internas (forget, input, output, cell).")

In [ ]:
print("=== LSTM Config-A: 32 unidades ===")
lstm_A, hist_lstmA, t_lstmA = entrenar(build_lstm(32), 'LSTM-A (32 units)')

In [ ]:
print("=== LSTM Config-B: 64 unidades ===")
lstm_B, hist_lstmB, t_lstmB = entrenar(build_lstm(64), 'LSTM-B (64 units)')

In [ ]:
plot_historia(hist_lstmA, 'LSTM-A 32 units', '#e67e22')
plot_historia(hist_lstmB, 'LSTM-B 64 units', '#e74c3c')

In [ ]:
# Seleccionar mejor LSTM
va_lstmA = max(hist_lstmA.history['val_accuracy'])
va_lstmB = max(hist_lstmB.history['val_accuracy'])

print(f"LSTM-A (32 units): mejor val_acc = {va_lstmA:.4f}")
print(f"LSTM-B (64 units): mejor val_acc = {va_lstmB:.4f}")

best_lstm       = lstm_A if va_lstmA >= va_lstmB else lstm_B
best_lstm_name  = 'LSTM-A (32 units)' if va_lstmA >= va_lstmB else 'LSTM-B (64 units)'
best_units_lstm = 32 if va_lstmA >= va_lstmB else 64
print(f"\nMejor LSTM: {best_lstm_name}")

In [ ]:
# Épocas con la mejor LSTM (mismo procedimiento que RNN)
EPOCH_LIST_LSTM = [3, 5, 10, 15, 20]
resultados_lstm = []
historiales_lstm = []

print(f"{'Épocas':>7} | train_acc | val_acc | val_loss")
print("-" * 45)

for ep in EPOCH_LIST_LSTM:
    m = build_lstm(best_units_lstm)
    m.compile(optimizer=keras.optimizers.Adam(0.001),
              loss='binary_crossentropy', metrics=['accuracy'])
    hist = m.fit(X_train, y_train, batch_size=BATCH_SIZE,
                 epochs=ep, validation_split=0.20, verbose=0)
    tr  = hist.history['accuracy'][-1]
    va  = hist.history['val_accuracy'][-1]
    vl  = hist.history['val_loss'][-1]
    print(f"{ep:>7} | {tr:.4f}    | {va:.4f}  | {vl:.4f}")
    historiales_lstm.append((ep, hist))
    resultados_lstm.append({'epochs': ep, 'train_acc': tr, 'val_acc': va})

idx_best_lstm = int(np.argmax([r['val_acc'] for r in resultados_lstm]))
BEST_EP_LSTM  = resultados_lstm[idx_best_lstm]['epochs']
print(f"\nMejor número de épocas LSTM: {BEST_EP_LSTM}")

---
## vi. Comparación RNN vs LSTM

In [ ]:
# Reentrenar cada modelo con su número óptimo de épocas para la evaluación final
print(f"Reentrenando RNN con {BEST_EP_RNN} épocas...")
rnn_final = build_rnn(best_units_rnn)
rnn_final.compile(optimizer=keras.optimizers.Adam(0.001),
                  loss='binary_crossentropy', metrics=['accuracy'])
t0_rnn = time.time()
hist_rnn_final = rnn_final.fit(X_train, y_train, batch_size=BATCH_SIZE,
                                epochs=BEST_EP_RNN, validation_split=0.20, verbose=0)
t_rnn_total = time.time() - t0_rnn
print(f"  Listo en {t_rnn_total:.0f}s")

print(f"\nReentrenando LSTM con {BEST_EP_LSTM} épocas...")
lstm_final = build_lstm(best_units_lstm)
lstm_final.compile(optimizer=keras.optimizers.Adam(0.001),
                   loss='binary_crossentropy', metrics=['accuracy'])
t0_lstm = time.time()
hist_lstm_final = lstm_final.fit(X_train, y_train, batch_size=BATCH_SIZE,
                                  epochs=BEST_EP_LSTM, validation_split=0.20, verbose=0)
t_lstm_total = time.time() - t0_lstm
print(f"  Listo en {t_lstm_total:.0f}s")

In [ ]:
# Gráfica comparativa en las mismas épocas
ep_comun = min(BEST_EP_RNN, BEST_EP_LSTM)

rnn_cmp = build_rnn(best_units_rnn)
rnn_cmp.compile(optimizer=keras.optimizers.Adam(0.001),
                loss='binary_crossentropy', metrics=['accuracy'])
hist_rnn_cmp = rnn_cmp.fit(X_train, y_train, batch_size=BATCH_SIZE,
                            epochs=ep_comun, validation_split=0.20, verbose=0)

lstm_cmp = build_lstm(best_units_lstm)
lstm_cmp.compile(optimizer=keras.optimizers.Adam(0.001),
                 loss='binary_crossentropy', metrics=['accuracy'])
hist_lstm_cmp = lstm_cmp.fit(X_train, y_train, batch_size=BATCH_SIZE,
                              epochs=ep_comun, validation_split=0.20, verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Comparación RNN vs LSTM — {ep_comun} épocas', fontsize=14, fontweight='bold')

for ax, met, lbl in zip(axes, ['accuracy', 'loss'], ['Exactitud', 'Pérdida']):
    ax.plot(hist_rnn_cmp.history[f'val_{met}'],  label='RNN — Val',  color='#3498db', linewidth=2)
    ax.plot(hist_lstm_cmp.history[f'val_{met}'], label='LSTM — Val', color='#e74c3c', linewidth=2)
    ax.set_xlabel('Época'); ax.set_ylabel(lbl)
    ax.set_title(f'{lbl} de Validación')
    ax.legend(fontsize=11); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('comparacion_RNN_LSTM.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Evaluación en test
y_pred_rnn  = (rnn_final.predict(X_test,  verbose=0) > 0.5).astype(int).flatten()
y_pred_lstm = (lstm_final.predict(X_test, verbose=0) > 0.5).astype(int).flatten()

acc_rnn   = accuracy_score(y_test, y_pred_rnn)
acc_lstm  = accuracy_score(y_test, y_pred_lstm)
f1_rnn    = f1_score(y_test, y_pred_rnn,  average='weighted')
f1_lstm   = f1_score(y_test, y_pred_lstm, average='weighted')

print("=" * 55)
print(f"{'':20s} {'RNN':>12} {'LSTM':>12}")
print("=" * 55)
print(f"{'Accuracy (test)':20s} {acc_rnn:>12.4f} {acc_lstm:>12.4f}")
print(f"{'F1-score (test)':20s} {f1_rnn:>12.4f} {f1_lstm:>12.4f}")
print(f"{'Tiempo (s)':20s} {t_rnn_total:>12.0f} {t_lstm_total:>12.0f}")
print(f"{'Épocas óptimas':20s} {BEST_EP_RNN:>12} {BEST_EP_LSTM:>12}")
print("=" * 55)

ganador = 'LSTM' if acc_lstm >= acc_rnn else 'RNN'
print(f"\nMejor modelo en test: {ganador}")

---
## vii. Métricas y Análisis Final

In [ ]:
CLASES_TEXTO = ['Negativa', 'Positiva']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Matrices de Confusión — Conjunto de Prueba (IMDB)', fontsize=14, fontweight='bold')

for ax, y_pred, titulo in zip(axes,
                               [y_pred_rnn, y_pred_lstm],
                               ['RNN Simple', 'LSTM']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASES_TEXTO, yticklabels=CLASES_TEXTO,
                linewidths=0.5, ax=ax, annot_kws={'size': 13})
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicho')

plt.tight_layout()
plt.savefig('confusion_matrices_RNN_LSTM.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
print("REPORTE DETALLADO — RNN Simple")
print("-" * 45)
print(classification_report(y_test, y_pred_rnn, target_names=CLASES_TEXTO))

print("\nREPORTE DETALLADO — LSTM")
print("-" * 45)
print(classification_report(y_test, y_pred_lstm, target_names=CLASES_TEXTO))

In [ ]:
# Visualización: barras de métricas comparativas
metricas  = ['Accuracy', 'F1-score']
vals_rnn  = [acc_rnn,  f1_rnn]
vals_lstm = [acc_lstm, f1_lstm]

x = np.arange(len(metricas))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - w/2, vals_rnn,  w, label='RNN Simple', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + w/2, vals_lstm, w, label='LSTM',       color='#e74c3c', edgecolor='white')

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(metricas, fontsize=12)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Valor', fontsize=12)
ax.set_title('Comparación de Métricas: RNN vs LSTM', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('comparacion_metricas.png', dpi=100, bbox_inches='tight')
plt.show()

---
## Conclusiones — Parte 2

### vi. Comparación RNN vs LSTM

| Criterio | RNN Simple | LSTM |
|----------|-----------|------|
| **Accuracy test** | reportado arriba | reportado arriba |
| **Tiempo por época** | menor (~4x más rápida) | mayor (más parámetros por compuertas) |
| **Textos largos** | degradación notable (gradiente se desvanece en secuencias >100 tokens) | mejor manejo gracias a las compuertas que filtran qué recordar |
| **Generalización** | riesgo de sobreajuste en textos largos | mejor generalización |

**¿Cuál tardó más?** La LSTM, porque cada celda tiene 4 matrices de pesos (compuertas forget, input, output + cell) frente a solo 1 en la SimpleRNN.

**¿La LSTM manejó mejor textos largos?** Sí. Las reseñas de IMDB tienen una mediana de ~240 palabras. La RNN simple sufre desvanecimiento del gradiente a esa longitud, perdiendo el contexto del inicio de la reseña. La LSTM lo mitiga con su mecanismo de memoria a largo plazo.

### vii. Factores que afectan el desempeño en texto

1. **Tamaño del vocabulario (`NUM_WORDS`):** un vocabulario pequeño pierde palabras discriminativas; uno muy grande aumenta el espacio de embedding y requiere más datos para entrenarlo. Con 10,000 palabras se cubre el ~95% del vocabulario activo de IMDB.

2. **Longitud máxima de secuencia (`MAX_LEN`):** si es muy corta se pierde contexto relevante (el final de la reseña, donde suele estar la valoración definitiva); si es muy larga aumenta el tiempo de cómputo y el riesgo de que la RNN pierda la información por gradiente desvanecido. `MAX_LEN=300` fue el balance óptimo.

3. **Número de unidades de la RNN/LSTM:** más unidades → mayor capacidad de memorizar patrones, pero también más parámetros → sobreajuste con datasets pequeños. **El hiperparámetro más sensible fue el número de épocas**: la diferencia entre 5 y 10 épocas produjo el mayor salto en val_accuracy, mientras que duplicar las unidades (32→64) tuvo un efecto menor.

### Dataset desbalanceado
**No:** IMDB tiene exactamente 12,500 positivas y 12,500 negativas en train. No se requirió ningún ajuste por desbalance.